In [4]:
import json

with open("./data/threaded_emails.json") as f:
    emails = json.load(f)

len(emails)

25306

In [5]:
documents = []

for email in emails:

    text = f"""
    Subject: {email.get("subject_clean","")}
    From: {email.get("from","")}
    To: {email.get("to","")}

    {email.get("body","")}
    """

    documents.append({
        "thread_id": email["thread_id"],
        "message_id": email.get("message_id",""),
        "text": text.strip()
    })

In [6]:
from rank_bm25 import BM25Okapi

corpus = [doc["text"].lower().split() for doc in documents]

bm25 = BM25Okapi(corpus)

In [14]:
def search(query, thread_id=None, top_k=5):

    query_tokens = query.lower().split()
    scores = bm25.get_scores(query_tokens)

    results = []

    for i, score in enumerate(scores):

        doc = documents[i]

        if thread_id and doc["thread_id"] != thread_id:
            continue

        results.append((score, doc))

    results.sort(key=lambda x: x[0], reverse=True)

    return results[:top_k]

In [15]:
results = search(
    "forecast",
    thread_id="T-001"
)

In [20]:
for score, doc in results:
    print(score)
    print(doc["text"])
    print("------")

8.13193839965843
Subject: mime-version: 1.0
    From: phillip.allen@enron.com
    To: jonathan.mckay@enron.com

    John,

Here is our North of Stanfield forecast for Jan.


Supply    Jan '01   Dec '00   Jan '00

 Sumas   900   910   815
 Jackson Pr.  125    33   223
 Roosevelt  300   298   333
 
 Total Supply  1325   1241   1371

Demand
 North of Chehalis 675   665   665
 South of Chehalis 650   575   706

 Total Demand  1325   1240   1371

Roosevelt capacity is 495.

Let me know how your forecast differs.


Phillip
------
8.13193839965843
Subject: mime-version: 1.0
    From: phillip.allen@enron.com
    To: jonathan.mckay@enron.com

    John,

Here is our North of Stanfield forecast for Jan.


Supply    Jan '01   Dec '00   Jan '00

 Sumas   900   910   815
 Jackson Pr.  125    33   223
 Roosevelt  300   298   333
 
 Total Supply  1325   1241   1371

Demand
 North of Chehalis 675   665   665
 South of Chehalis 650   575   706

 Total Demand  1325   1240   1371

Roosevelt capacity is 49

In [21]:
print(len(documents))
print(type(bm25))

25306
<class 'rank_bm25.BM25Okapi'>
